In [1]:
## conda activate basicEnvMM
import cv2
import numpy as np
from skimage import io, transform, feature, filters
from matplotlib import pyplot as plt



In [3]:
roi_path = '/mnt/scratch1/maycon/Diego_RSV_CosMx/Round_2/tmp/ROI.png'
fov_path = '/mnt/scratch1/maycon/Diego_RSV_CosMx/Round_2/tmp/FOV.png'

In [ ]:
# I think the problem here is with the images format ... I tried to get it from a screenshoot 

### chatBox inputs 

In [ ]:
# there're two images.
# ROI.png and FOV.png

# They are TMAs images. They have similar shapes

# But
# 1. they are not aligned yet
# 2. there're some marks in the ROI (squares) which I want to know where they are in the FOV
# 3. I still want to keep what ever information I got on FOV + the squares marks

# Obs: The ROI image have the squares, the image itself, and some text elements.

### chatBox outs 

In [2]:
# This pipeline does the following:

# 1. Loads the ROI and FOV images.

# 2. Preprocesses the images by converting them to grayscale.

# 3. Registers the ROI image to the FOV image using feature-based registration.

# 4. Detects square marks in the aligned ROI image.

# 5. Transfers the detected squares to the FOV image.

# 6. Displays the original ROI, original FOV, and the result with transferred squares.

# To align the ROI.png and FOV.png images and transfer the square marks from ROI to FOV while preserving the FOV information, you'll need to follow these steps:

# Image Registration
# First, you'll need to align the two TMA (Tissue Microarray) images using image registration techniques. This process will find the optimal rotation and translation to match the ROI.png to FOV.png3.

# Mark Transfer
# After registration, you can identify the locations of the square marks in the ROI image and transfer them to the corresponding positions in the FOV image.

# Information Preservation
# The original FOV image information will be kept intact, with the square marks added as an overlay or in a separate channel.

# Here's a more detailed approach:

# Image Registration
# Use a feature-based registration algorithm to align ROI.png with FOV.png. This method works well for images with similar structures1.

# Apply a cost function weighting to focus on the TMA spots and ignore text elements in the ROI image2. This will improve registration accuracy by concentrating on the relevant image features.

# Perform the registration using a multi-stage approach:

# Start with a coarse alignment using the entire image.

# Refine the alignment using smaller regions of interest around the TMA spots4.

# Mark Transfer and Information Preservation
# After registration, identify the square marks in the ROI image using image segmentation techniques6.

# Create a new layer or channel in the FOV image to store the square mark information.

# Transfer the identified square marks to the new layer in the FOV image, using the transformation parameters obtained from the registration step.

# Preserve all original FOV image data by keeping it in separate channels or layers.

# Final Result
# The final output will be a multi-channel or multi-layer image containing:

# The original FOV image data

# The aligned ROI image (optional)

# A layer with the transferred square marks

# This approach allows you to maintain all the information from the FOV image while adding the square marks from the ROI image in their correct positions78.

# To visualize the results, you can overlay the square marks on the FOV image or switch between different layers/channels as needed

def load_images(roi_path, fov_path):
    roi = cv2.imread(roi_path)
    fov = cv2.imread(fov_path)
    return roi, fov

def preprocess_images(roi, fov):
    roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    fov_gray = cv2.cvtColor(fov, cv2.COLOR_BGR2GRAY)
    return roi_gray, fov_gray

def register_images(roi_gray, fov_gray):
    # Feature detection
    orb = cv2.ORB_create()
    kp1, des1 = orb.detectAndCompute(roi_gray, None)
    kp2, des2 = orb.detectAndCompute(fov_gray, None)

    # Feature matching
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)
    matches = sorted(matches, key=lambda x: x.distance)

    # Find homography
    src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

    # Warp ROI image
    h, w = fov_gray.shape
    roi_aligned = cv2.warpPerspective(roi_gray, M, (w, h))
    
    return roi_aligned, M

def detect_squares(roi_aligned):
    # Apply threshold
    _, binary = cv2.threshold(roi_aligned, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # Find contours
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    squares = []
    for cnt in contours:
        # Approximate contour to polygon
        approx = cv2.approxPolyDP(cnt, 0.04 * cv2.arcLength(cnt, True), True)
        if len(approx) == 4:  # If it's a quadrilateral
            x, y, w, h = cv2.boundingRect(approx)
            aspect_ratio = float(w)/h
            if 0.9 <= aspect_ratio <= 1.1:  # Check if it's square-like
                squares.append(approx)
    
    return squares

def transfer_squares(fov, squares, M):
    result = fov.copy()
    for square in squares:
        # Transform square coordinates
        transformed_square = cv2.perspectiveTransform(square.astype(np.float32), M)
        # Draw square on FOV image
        cv2.drawContours(result, [transformed_square.astype(int)], 0, (0, 255, 0), 2)
    
    return result




In [3]:
def main():
    # Load images
    roi, fov = load_images('ROI.png', 'FOV.png')
    
    # Preprocess images
    roi_gray, fov_gray = preprocess_images(roi, fov)
    
    # Register images
    roi_aligned, M = register_images(roi_gray, fov_gray)
    
    # Detect squares in ROI
    squares = detect_squares(roi_aligned)
    
    # Transfer squares to FOV
    result = transfer_squares(fov, squares, M)
    
    # Display results
    plt.figure(figsize=(15, 5))
    plt.subplot(131), plt.imshow(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)), plt.title('ROI')
    plt.subplot(132), plt.imshow(cv2.cvtColor(fov, cv2.COLOR_BGR2RGB)), plt.title('FOV')
    plt.subplot(133), plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB)), plt.title('Result')
    plt.show()

if __name__ == "__main__":
    main()

[ WARN:0@141.253] global loadsave.cpp:241 findDecoder imread_('ROI.png'): can't open/read file: check file path/integrity
[ WARN:0@141.253] global loadsave.cpp:241 findDecoder imread_('FOV.png'): can't open/read file: check file path/integrity


error: OpenCV(4.10.0) /croot/opencv-suite_1738943342777/work/modules/imgproc/src/color.cpp:196: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'


In [14]:
roi, fov = load_images('ROI.png', 'FOV.png')
#roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)


[ WARN:0@263.483] global loadsave.cpp:241 findDecoder imread_('ROI.png'): can't open/read file: check file path/integrity
[ WARN:0@263.483] global loadsave.cpp:241 findDecoder imread_('FOV.png'): can't open/read file: check file path/integrity


In [13]:
roi